# 5-Hop Reasoning Pipeline Benchmark

Run the **5-hop reasoning pipeline** on a subset of the single-article dataset, compute the same metrics as in Phase 2 (accuracy, F1 macro, precision/recall macro), and compare to the baselines from **notebooks/02_evaluation.ipynb**.

- **First run**: 100 samples only (to validate and check metrics quickly).
- **Metrics**: Same as 02_evaluation (accuracy, f1_macro, precision_macro, recall_macro).
- **Comparison**: On the **same 100 samples**, we compare 5-hop predictions to FinBERT and GPT baselines (using existing prediction columns in the CSV).

## 1. Setup and load 100 samples

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path().resolve().parent))

from src.utils.data_loader import load_all_dataframes
from src.evaluation.metrics import compute_classification_metrics, confusion_matrix_dict

base = Path().resolve().parent
data = load_all_dataframes(base)
sa = data["single_article"]

# Use first 100 samples for this benchmark (same order as in CSV)
N_SAMPLES = 10
sample = sa.head(N_SAMPLES).copy()
print(f"Loaded {len(sample)} samples. Columns: text, ticker, true_sentiment")
sample[["text", "ticker", "true_sentiment"]].head(3)

Loaded ground truth: 2291 rows
Loaded single article predictions: 2291 rows
Loaded all-day articles: 293 rows
Loaded 10 samples. Columns: text, ticker, true_sentiment


,text,ticker,true_sentiment
0,The Euro was able to appreciate particularly s...,EURCHF,1
1,EUR/CHF yesterday broke above 1.00. Economists...,EURCHF,1
2,EUR/CHF vaults parity for the first time since...,EURCHF,0


## 2. Initialize 5-hop pipeline

Requires `OPENAI_API_KEY` in the environment (e.g. from `.env` or `export OPENAI_API_KEY=...`).

In [2]:
from src.pipeline import ReasoningPipeline
from src.pipeline.llm_client import LLMClient

llm = LLMClient(
    max_tokens=128,
    model="phi3",
)

pipeline = ReasoningPipeline(llm_client=llm)

## 3. Run 5-hop pipeline on 100 samples

For each row we call `pipeline.run(text, ticker=ticker)` and map Hop 4 sentiment (`"Positive"`/`"Negative"`/`"Neutral"`) to numeric labels 1 / -1 / 0 to match the dataset.

In [3]:
def sentiment_str_to_numeric(s: str) -> float:
    """Map pipeline sentiment to dataset labels: Positive->1, Negative->-1, Neutral->0."""
    if s is None:
        return np.nan
    s = (s or "").strip().lower()
    if s == "positive":
        return 1.0
    if s == "negative":
        return -1.0
    if s == "neutral":
        return 0.0
    return np.nan


y_true_5hop = sample["true_sentiment"].values.astype(float)
y_pred_5hop = np.full(len(sample), np.nan, dtype=float)

if pipeline is not None:
    for idx in range(len(sample)):
        row = sample.iloc[idx]
        # text = row.get("text") or row.get("title") or ""
        text = row.get("title")
        ticker = row.get("ticker")
        if pd.isna(ticker):
            ticker = None
        else:
            ticker = str(ticker).strip() or None
        try:
            context = pipeline.run(text, ticker=ticker)
            print(context)
            print("*************")
            sent = context.sentiment
            y_pred_5hop[idx] = sentiment_str_to_numeric(sent)
        except Exception as e:
            print(f"Sample {idx}: {e}")
            y_pred_5hop[idx] = np.nan
    print(
        f"Completed. Valid predictions: {np.sum(np.isfinite(y_pred_5hop))} / {len(sample)}"
    )
else:
    print("Pipeline not initialized; skipping run.")

ReasoningContext(text='Euro to benefit from the ECBs pronounced hawkish determination – Commerzbank', ticker='EURCHF', entities=['FROM', 'EURO', 'ECBS', 'THE'], primary_entity='EURO', financial_aspects=[], primary_aspect=None, implicit_cues=[], cue_types=[], sentiment='Positive', sentiment_score=0.8, sentiment_reasoning="The headline 'Euro to benefit from the ECBs pronounced hawkish determination' suggests a positive sentiment for the Euro (base currency) relative to its quote currency. The term 'benefit' indicates that the news is favorable and likely leads to an appreciation of the base currency, which in this case is the Euro.", market_implication='Bullish', market_reasoning='Fallback mapping from sentiment', hop_results={'entity_grounding': {'entities': ['FROM', 'EURO', 'ECBS', 'THE'], 'primary_entity': 'EURO', 'confidence': 'low', 'reasoning': 'Fallback pattern matching'}, 'financial_aspect': {'aspects': [], 'primary_aspect': None, 'reasoning': 'Keyword-based fallback'}, 'implicit

## 4. Benchmark metrics for 5-hop

Same metrics as **02_evaluation**: accuracy, F1 macro, precision macro, recall macro. Invalid predictions (NaN) are dropped when computing metrics.

In [4]:
metrics_5hop = compute_classification_metrics(y_true_5hop, y_pred_5hop)

if "error" in metrics_5hop:
    print("Error:", metrics_5hop["error"])
else:
    print("5-Hop Pipeline — Metrics (same 100 samples)")
    print("-" * 50)
    print(f"  n (valid pairs): {metrics_5hop['n']}")
    print(f"  Accuracy:        {metrics_5hop['accuracy']:.4f}")
    print(f"  F1 (macro):     {metrics_5hop['f1_macro']:.4f}")
    print(f"  Precision (macro): {metrics_5hop['precision_macro']:.4f}")
    print(f"  Recall (macro):   {metrics_5hop['recall_macro']:.4f}")

# Confusion matrix
if "error" not in metrics_5hop:
    cm_5hop = confusion_matrix_dict(y_true_5hop, y_pred_5hop)
    print("\nConfusion matrix (rows=true, cols=pred):")
    print(
        pd.DataFrame(
            cm_5hop["matrix"], index=cm_5hop["labels"], columns=cm_5hop["labels"]
        )
    )

5-Hop Pipeline — Metrics (same 100 samples)
--------------------------------------------------
  n (valid pairs): 10
  Accuracy:        0.4000
  F1 (macro):     0.4508
  Precision (macro): 0.5667
  Recall (macro):   0.4444

Confusion matrix (rows=true, cols=pred):
          Negative  Neutral  Positive
Negative         1        0         1
Neutral          0        1         1
Positive         0        4         2


## 5. Compare to baselines (same 100 samples)

On the **same 100 rows**, we evaluate the baseline models (FinBERT, GPT-P1–P4, etc.) using the prediction columns already in the CSV. This gives a fair comparison: 5-hop vs baselines on the same subset.

In [5]:
from src.evaluation.comparison import evaluate_single_article_models

# Evaluate all baselines on the same 100 samples
baseline_results = evaluate_single_article_models(sample, y_true_col="true_sentiment")

# Build comparison table: model, accuracy, f1_macro, precision_macro, recall_macro
rows = []
# 5-hop (our run)
if "error" not in metrics_5hop:
    rows.append(
        {
            "model": "5hop_pipeline",
            "accuracy": metrics_5hop["accuracy"],
            "f1_macro": metrics_5hop["f1_macro"],
            "precision_macro": metrics_5hop["precision_macro"],
            "recall_macro": metrics_5hop["recall_macro"],
        }
    )
# Baselines
for name, res in baseline_results.items():
    if "metrics" in res and "error" not in res["metrics"]:
        m = res["metrics"]
        rows.append(
            {
                "model": name,
                "accuracy": m["accuracy"],
                "f1_macro": m["f1_macro"],
                "precision_macro": m["precision_macro"],
                "recall_macro": m["recall_macro"],
            }
        )

df_compare = pd.DataFrame(rows).round(4)
print("Comparison on same 100 samples (5-hop vs baselines from CSV)")
print("-" * 60)
display(df_compare)

Comparison on same 100 samples (5-hop vs baselines from CSV)
------------------------------------------------------------


,model,accuracy,f1_macro,precision_macro,recall_macro
0,5hop_pipeline,0.4,0.4508,0.5667,0.4444
1,finbert,0.2,0.1619,0.3750,0.2222
2,finbert_a,0.8,0.7475,0.8333,0.7778
3,gpt_p1,0.8,0.7475,0.8333,0.7778
4,gpt_p2,0.8,0.6966,0.7857,0.6667
5,gpt_p3,0.9,0.8222,0.8889,0.8333
6,gpt_p4,0.8,0.5556,0.5000,0.6667
7,gpt_p7,0.7,0.4410,0.3968,0.5000
8,gpt_p1n,0.9,0.8632,0.9524,0.8333
9,gpt_p2n,0.7,0.4889,0.5556,0.5000


## 6. Discussion

**Observed result (on this 100-sample benchmark):** The 5-hop pipeline typically **outperforms FinBERT** but **lags behind the single-shot GPT baselines** (e.g. GPT-P2, GPT-P4).

**Possible reasons:**

1. **Task fit**  
   The single-shot GPT prompts (P1–P4) are tuned for **direct sentiment classification** on headlines. Many headlines in the dataset may be relatively explicit, so “one prompt → one label” works well. The 5-hop pipeline targets **implicit** sentiment (entity → aspect → cues → sentiment → market), which adds structure but also more steps where small errors can propagate.

2. **Calibration and prompt design**  
   The GPT baselines were likely optimized (or chosen) for this dataset/task. The 5-hop prompts are more general and chain multiple hops; the final sentiment hop may be less aligned with the annotation scheme (e.g. Positive/Negative/Neutral) than a single, task-specific prompt.

3. **Variance on 100 samples**  
   With only 100 samples, differences can be noisy. Running on a larger subset (e.g. 500 or full dataset) would give a more stable comparison.

4. **Where 5-hop may still add value**  
   The pipeline is designed for **implicit** and **explainable** reasoning (entity, aspect, cues, sentiment, market implication). Even if accuracy is lower than single-shot GPT on this benchmark, it may be more useful when interpretability or structured reasoning matters, or on data where sentiment is more implicit.

**Next steps (optional):** Error analysis (where does 5-hop disagree with GPT or ground truth?), prompt tuning for the sentiment hop, or running the same comparison on a larger sample.

## 7. Relation to 02_evaluation.ipynb

- **02_evaluation** reports metrics on the **full** single-article dataset (~2291 rows) for FinBERT and GPT baselines.
- **This notebook** reports metrics on **100 samples** for both 5-hop and the same baselines, so the comparison above is **fair** (same subset).
- To compare 5-hop (100 samples) to **full-dataset** baseline numbers: run 02_evaluation and compare the baseline table there to the 5-hop row above, keeping in mind the different sample sizes (100 vs 2291).